In [21]:
import sys
import os

PROJECT_ROOT = os.path.abspath("../..")
sys.path.append(PROJECT_ROOT)

import time
import pandas as pd
from Scripts.corss_validation.cross_val import cross_validate
from Scripts.models.logistic_regression import LogisticRegressionModel

In [22]:
# load data
df = pd.read_csv("../../Data/pre_split_features.csv")
# dataset sampling for faster execution during development
df = df.sample(n=4000,random_state=42).reset_index(drop=True)

In [23]:
param_grid = {
    "C": [0.01, 0.1, 1]
}

start_time = time.time()
print("\n=== Base engineered features ===")
results = cross_validate(
    model_class=LogisticRegressionModel,
    df=df,
    param_grid=param_grid,
    n_outer=3,
    n_inner=2,
    scoring="f1_macro",
    use_interactions=False
)
elapsed = time.time() - start_time
print(f"Execution time: {elapsed:.2f} sec")



=== Base engineered features ===
Samples: 4000, Features: 74
fold 1/3 f1_macro=0.2451 params={'C': 0.01}
fold 2/3 f1_macro=0.2475 params={'C': 0.01}
fold 3/3 f1_macro=0.2530 params={'C': 0.01}

results:
  accuracy: 0.3488 +/- 0.0150
  precision_macro: 0.2629 +/- 0.0024
  recall_macro: 0.2743 +/- 0.0060
  f1_macro: 0.2485 +/- 0.0033
  train_f1: 0.3103 +/- 0.0020
  train_test_gap: 0.0617 +/- 0.0047
Execution time: 1.13 sec


In [24]:
start_time = time.time()

print("\n=== Enhanced behavioural features ===")
results_enhanced = cross_validate(
    model_class=LogisticRegressionModel,
    df=df,
    param_grid=param_grid,
    n_outer=3,
    n_inner=2,
    scoring="f1_macro",
    use_interactions=True
)

elapsed = time.time() - start_time
print(f"Execution time: {elapsed:.2f} sec")


=== Enhanced behavioural features ===
Samples: 4000, Features: 100
fold 1/3 f1_macro=0.2469 params={'C': 0.01}
fold 2/3 f1_macro=0.2478 params={'C': 0.01}
fold 3/3 f1_macro=0.2364 params={'C': 0.01}

results:
  accuracy: 0.3315 +/- 0.0085
  precision_macro: 0.2646 +/- 0.0066
  recall_macro: 0.2670 +/- 0.0052
  f1_macro: 0.2437 +/- 0.0052
  train_f1: 0.3195 +/- 0.0041
  train_test_gap: 0.0758 +/- 0.0092
Execution time: 3.89 sec


In [25]:
# save results
pd.DataFrame(results["fold_metrics"]).to_csv("../../Results/fold_metrics_lr.csv",index=False)
pd.DataFrame(results["best_params"]).to_csv("../../Results/best_params_lr.csv",index=False)

pd.DataFrame(results_enhanced["fold_metrics"]).to_csv("../../Results/fold_metrics_lr_enhanced.csv",index=False)
pd.DataFrame(results_enhanced["best_params"]).to_csv("../../Results/best_params_lr_enhanced.csv",index=False)

print("\nClass distribution:")
print(df["review_score"].value_counts(normalize=True))


Class distribution:
review_score
5    0.59575
4    0.20075
1    0.09600
3    0.07675
2    0.03075
Name: proportion, dtype: float64
